# 딥러닝 3일차 (1) — 손실 함수 (MSE · 교차 엔트로피)

수업 메모 `딥러닝_노트_수정본` 의 손실 · 교차 엔트로피 부분을 메모의 순서 · 예시 그대로 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `딥러닝` 교재8 p.41~45 · 49 — 손실 함수의 개념 · 평균 제곱 오차 · 교차 엔트로피 오차 · 이진 / 다중 분류 · 손실 함수 실습. p.46~48 배치 vs 미니배치는 아직 배우지 않아 제외)

- 출력층 설계(항등 · 시그모이드 · 소프트맥스)는 [Day 032 `neural_network_forward.ipynb`](../../2026-10-23/lecture/neural_network_forward.ipynb) 에 있습니다.
- 미분 · 수치 미분은 [`numerical_differentiation.ipynb`](./numerical_differentiation.ipynb) 에 이어서 정리했습니다.
- 메모 · 교재에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

In [1]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # 리눅스 · 코랩 : 설치된 한글 폰트가 있으면 사용
    names = {f.name for f in fm.fontManager.ttflist}
    for name in ['NanumGothic', 'Noto Sans CJK KR']:
        if name in names:
            plt.rcParams['font.family'] = name
            break
plt.rcParams['axes.unicode_minus'] = False

In [2]:
import os
import numpy as np
from matplotlib.animation import FuncAnimation, PillowWriter

os.makedirs('./images', exist_ok=True)
np.set_printoptions(precision=4, suppress=True)

## 1. 손실(Loss)이란 (교재 p.41)

- **손실** = 모델의 예측값 $y$ 와 정답 $t$ 의 **차이의 크기**. 가중치 $W$ · 편향 $b$ 가 아직 데이터에 맞춰지지 않아서 생긴다.
- **손실 함수** = 오차 여러 개를 **숫자(스칼라) 하나**로 바꿔 "모델이 지금 얼마나 **나쁜가**" 를 나타내는 함수.
- **학습** = 손실 함수의 값이 **0 에 가까워지는** 가중치 · 편향을 찾는 것.

| 문제 | 출력층 | 손실 함수 |
|---|---|---|
| 회귀 (집값) | 항등 함수 · 뉴런 1개 | **평균 제곱 오차 (MSE)** |
| 이진 분류 (스팸) | 시그모이드 · 뉴런 1개 | **이진 교차 엔트로피 (BCE)** |
| 다중 분류 (숫자 0~9) | 소프트맥스 · 뉴런 = 클래스 수 | **교차 엔트로피 (CEE)** |

In [3]:
t = np.array([3.0, 5.0, 2.0, 7.0])     # 정답 (예: 집값, 억 원)
y = np.array([2.5, 5.5, 2.0, 9.0])     # 모델의 예측값

error = y - t
print("오차 y - t :", error, "  ← 숫자가 4개라 '얼마나 나쁜지' 한눈에 비교하기 어렵다")
print("손실(MSE)  :", np.mean(error ** 2), "  ← 숫자 하나")

오차 y - t : [-0.5  0.5  0.   2. ]   ← 숫자가 4개라 '얼마나 나쁜지' 한눈에 비교하기 어렵다
손실(MSE)  : 1.125   ← 숫자 하나


## 2. 회귀의 손실 함수 — 평균 제곱 오차 MSE (교재 p.42)

$$E = \frac{1}{n}\sum_{i=1}^{n}(y_i - t_i)^2$$

**오차 → 제곱 → 모두 더하기 → 개수로 나누기(평균)**. 데이터가 1개면 그냥 **제곱 오차**이고, 여러 개의 제곱 오차를 **평균** 낸 것이 MSE 다.

In [4]:
def mean_squared_error(y, t):
    y, t = np.asarray(y, dtype=float), np.asarray(t, dtype=float)
    return np.mean((y - t) ** 2)

from sklearn.metrics import mean_squared_error as sk_mse

print("직접 만든 MSE :", mean_squared_error(y, t))
print("sklearn MSE   :", sk_mse(t, y))          # sklearn 은 (정답, 예측) 순서
print("RMSE          :", np.sqrt(mean_squared_error(y, t)), " ← 단위를 정답과 같게(억 원)")

직접 만든 MSE : 1.125
sklearn MSE   : 1.125
RMSE          : 1.0606601717798212  ← 단위를 정답과 같게(억 원)


**'제곱' 의 두 가지 의미**

1. **부호 제거** — 정답보다 크게 틀린 것(+)과 작게 틀린 것(−)이 서로 **상쇄되지 않는다**.
2. **큰 오차에 더 큰 벌점** — 오차가 2배면 손실은 **4배**, 10배면 **100배**.

In [5]:
# 1) 부호 제거 : 그냥 평균내면 +2 와 -2 가 상쇄되어 '오차 0' 처럼 보인다
e = np.array([2.0, -2.0])
print("오차 평균      :", e.mean(), " ← 틀렸는데 0")
print("제곱 오차 평균 :", (e ** 2).mean())

# 2) 큰 오차에 더 큰 벌점 (제곱 = 2차 함수)
for err in [1, 2, 3, 10]:
    print(f"오차 {err:>2} → 절댓값 {err:>2} · 제곱 {err ** 2:>3}")

# 오차가 같은 총합이라도 한 곳에 몰리면 MSE 가 훨씬 크다
spread = np.array([1.0, 1.0, 1.0, 1.0])   # 골고루 1씩
peak   = np.array([0.0, 0.0, 0.0, 4.0])   # 한 개만 4
print("\n오차 합 4 를 골고루 :", "MAE", np.abs(spread).mean(), "· MSE", (spread ** 2).mean())
print("오차 합 4 를 한 곳에 :", "MAE", np.abs(peak).mean(), "· MSE", (peak ** 2).mean(), " ← MSE 는 치명적인 오차 하나를 크게 벌준다")

오차 평균      : 0.0  ← 틀렸는데 0
제곱 오차 평균 : 4.0
오차  1 → 절댓값  1 · 제곱   1
오차  2 → 절댓값  2 · 제곱   4
오차  3 → 절댓값  3 · 제곱   9
오차 10 → 절댓값 10 · 제곱 100

오차 합 4 를 골고루 : MAE 1.0 · MSE 1.0
오차 합 4 를 한 곳에 : MAE 1.0 · MSE 4.0  ← MSE 는 치명적인 오차 하나를 크게 벌준다


In [6]:
# 🎬 애니메이션 : 예측값 하나를 정답에서 멀리 옮길 때 — 절댓값 오차(선) vs 제곱 오차(정사각형 넓이)
t0 = 2.0
offsets = np.concatenate([np.linspace(0, 4, 33), np.full(6, 4.0)])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={'width_ratios': [1.15, 1]})

def draw(i):
    for ax in axes: ax.clear()
    d = offsets[i]; yp = t0 + d
    ax = axes[0]
    ax.add_patch(plt.Rectangle((t0, 0), d, d, color='tab:red', alpha=0.25))
    ax.plot([t0, yp], [0, 0], color='tab:blue', lw=5)
    ax.plot(t0, 0, 'ko', ms=10); ax.plot(yp, 0, 'o', color='tab:red', ms=10)
    ax.text(t0, -0.45, '정답 t', ha='center'); ax.text(yp, -0.45, '예측 y', ha='center', color='tab:red')
    ax.text(t0 + d / 2, d / 2, f'제곱\n{d ** 2:.1f}', ha='center', va='center', fontsize=12, color='darkred')
    ax.set_xlim(1.5, 6.6); ax.set_ylim(-0.8, 4.4); ax.set_aspect('equal'); ax.set_yticks([])
    ax.set_title(f'오차 {d:.2f} → 제곱 오차 = 정사각형 넓이 {d ** 2:.2f}')
    ax = axes[1]
    xs = np.linspace(0, 4, 200)
    ax.plot(xs, xs, color='tab:blue', lw=2, label='절댓값 오차 |y−t|')
    ax.plot(xs, xs ** 2, color='tab:red', lw=2, label='제곱 오차 (y−t)²')
    ax.plot(d, d, 'o', color='tab:blue'); ax.plot(d, d ** 2, 'o', color='tab:red')
    ax.set_xlabel('오차 크기'); ax.set_ylabel('손실'); ax.set_ylim(0, 16.5); ax.grid(alpha=0.3); ax.legend(loc='upper left')
    ax.set_title('오차 2배 → 제곱 오차 4배')
    fig.tight_layout()

ani = FuncAnimation(fig, draw, frames=len(offsets), interval=120)
ani.save('./images/anim_01_mse_square.gif', writer=PillowWriter(fps=8), dpi=72)
plt.close()
print("저장 : ./images/anim_01_mse_square.gif")

저장 : ./images/anim_01_mse_square.gif


![MSE 제곱 오차 애니메이션](./images/anim_01_mse_square.gif)

🎬 예측값을 정답에서 0 → 4 만큼 멀리 옮겼다. 절댓값 오차는 **직선**으로 늘지만, 제곱 오차는 한 변이 오차인 **정사각형의 넓이**라서 **곡선(2차)** 으로 늘어난다.

> **📝 고친 부분** (교재 p.42 · 메모)
> - "오차가 클수록 손실 값이 **기하급수적으로** 커진다" → 기하급수(지수 함수, $2^x$)가 아니라 **제곱(2차 함수, $x^2$)** 으로 커진다. 오차 10 → 손실 100 이지 $2^{10}$ = 1024 가 아니다.
> - 메모 `MSE: L = (정답 - y)²` → 데이터 **1개**의 식이라 **제곱 오차**다. MSE 는 여러 개의 제곱 오차를 **평균** 낸 것(데이터 1개면 값은 같다).

## 3. 분류의 손실 함수 — 교차 엔트로피 오차 CEE (교재 p.43, 45)

$$E = -\sum_{k} t_k \log y_k$$

- $y_k$ : 소프트맥스 출력 = $k$ 번째 클래스일 **예측 확률** $(0 < y_k < 1)$
- $t_k$ : **원-핫 인코딩** 된 정답 (정답 클래스만 1, 나머지 0)
- $\log$ 는 **자연로그** $\ln$ (밑이 $e \approx 2.718$, 파이썬 `np.log`)

정답이 아닌 클래스는 $t_k = 0$ 이 곱해져서 사라진다 → **정답 클래스의 확률 하나만** 보고 $-\log(\text{정답 확률})$ 을 계산하는 것과 같다.

In [7]:
# log 의 특징 : log 1 = 0, 0 ~ 1 사이면 음수, 0 에 가까울수록 -무한대
for p in [1.0, 0.9, 0.7, 0.5, 0.1, 0.01]:
    print(f"정답 확률 {p:<4} → log = {np.log(p):7.3f} → -log = {-np.log(p) + 0.0:6.3f}")

with np.errstate(divide='ignore'):
    print("\nnp.log(0) =", np.log(0.0), " ← 확률이 0 이면 계산이 망가진다")

정답 확률 1.0  → log =   0.000 → -log =  0.000
정답 확률 0.9  → log =  -0.105 → -log =  0.105
정답 확률 0.7  → log =  -0.357 → -log =  0.357
정답 확률 0.5  → log =  -0.693 → -log =  0.693
정답 확률 0.1  → log =  -2.303 → -log =  2.303
정답 확률 0.01 → log =  -4.605 → -log =  4.605

np.log(0) = -inf  ← 확률이 0 이면 계산이 망가진다


In [8]:
def cross_entropy_error(y, t):
    """y : 예측 확률(소프트맥스 출력), t : 원-핫 정답. 1개 또는 여러 개 모두 가능"""
    y, t = np.asarray(y, dtype=float), np.asarray(t, dtype=float)
    if y.ndim == 1:                       # 데이터 1개 → (1, 클래스 수) 로 맞추기
        y, t = y.reshape(1, -1), t.reshape(1, -1)
    delta = 1e-7                          # log(0) = -inf 를 막는 아주 작은 값
    return -np.sum(t * np.log(y + delta)) / y.shape[0]    # 데이터 개수로 나눠 평균

# 교재 p.45 예시 : [개, 고양이, 호랑이], 정답 = 고양이
t = np.array([0, 1, 0])
y_good = np.array([0.1, 0.7, 0.2])       # 고양이에 70%
y_bad  = np.array([0.7, 0.1, 0.2])       # 개에 70% (오답)

print("잘 맞힘 :", round(cross_entropy_error(y_good, t), 3), " = -log 0.7")
print("틀림    :", round(cross_entropy_error(y_bad, t), 3), " = -log 0.1")
print("확률 0  :", round(cross_entropy_error([1.0, 0.0, 0.0], t), 3), " ← delta 덕분에 inf 대신 큰 값(-log 1e-7)")

# 메모 예시 : 원-핫 [0, 0, 1, 0, 0], 정답 확률 P3 = 0.7 (클래스 5개 → 출력 뉴런 5개)
t5 = np.array([0, 0, 1, 0, 0])
y5 = np.array([0.05, 0.1, 0.7, 0.1, 0.05])
print("\n메모 예시 (5클래스) :", round(cross_entropy_error(y5, t5), 3), " = -log 0.7 ≈ 0.36 — 오답 클래스 확률은 0 이 곱해져 빠진다")

잘 맞힘 : 0.357  = -log 0.7
틀림    : 2.303  = -log 0.1
확률 0  : 16.118  ← delta 덕분에 inf 대신 큰 값(-log 1e-7)

메모 예시 (5클래스) : 0.357  = -log 0.7 ≈ 0.36 — 오답 클래스 확률은 0 이 곱해져 빠진다


In [9]:
# 원-핫 대신 '정답 번호' 로도 계산할 수 있다 — 정답 클래스의 확률만 꺼내면 된다
def cross_entropy_error_label(y, label):
    y = np.atleast_2d(np.asarray(y, dtype=float))
    label = np.atleast_1d(label)
    return -np.mean(np.log(y[np.arange(len(y)), label] + 1e-7))

# 데이터 3개 (고양이, 개, 호랑이)
Y = np.array([[0.1, 0.7, 0.2],
              [0.8, 0.1, 0.1],
              [0.3, 0.3, 0.4]])
labels = np.array([1, 0, 2])
T = np.eye(3)[labels]                     # 정답 번호 → 원-핫

from sklearn.metrics import log_loss
print("원-핫 정답   :", round(cross_entropy_error(Y, T), 4))
print("정답 번호    :", round(cross_entropy_error_label(Y, labels), 4))
print("sklearn      :", round(log_loss(labels, Y), 4), " ← 세 값이 같다 (3개의 평균)")

원-핫 정답   : 0.4987
정답 번호    : 0.4987
sklearn      : 0.4987  ← 세 값이 같다 (3개의 평균)


In [10]:
# 🎬 애니메이션 : 정답(고양이) 확률이 0.05 → 0.98 로 오를 때 교차 엔트로피
ps = np.concatenate([np.linspace(0.05, 0.98, 40), np.full(6, 0.98)])
classes = ['개', '고양이(정답)', '호랑이']
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

def draw(i):
    for ax in axes: ax.clear()
    p = ps[i]; rest = (1 - p) / 2
    probs = [rest * 1.2, p, rest * 0.8]
    loss = -np.log(p)
    ax = axes[0]
    ax.bar(classes, probs, color=['lightgray', 'tab:red', 'lightgray'])
    for k, v in enumerate(probs): ax.text(k, v + 0.02, f'{v:.2f}', ha='center')
    ax.set_ylim(0, 1.1); ax.set_title('소프트맥스 출력 y (합 = 1)'); ax.set_ylabel('예측 확률')
    ax = axes[1]
    xs = np.linspace(0.02, 1, 300)
    ax.plot(xs, -np.log(xs), color='tab:purple', lw=2)
    ax.plot(p, loss, 'o', color='tab:red', ms=10)
    ax.axhline(0, color='gray', lw=0.6)
    ax.set_xlabel('정답 클래스의 예측 확률'); ax.set_ylabel('손실 = −log(정답 확률)')
    ax.set_ylim(-0.1, 4); ax.grid(alpha=0.3)
    ax.set_title(f'교차 엔트로피 = −log({p:.2f}) = {loss:.3f}')
    fig.tight_layout()

ani = FuncAnimation(fig, draw, frames=len(ps), interval=120)
ani.save('./images/anim_02_cross_entropy.gif', writer=PillowWriter(fps=8), dpi=72)
plt.close()
print("저장 : ./images/anim_02_cross_entropy.gif")

저장 : ./images/anim_02_cross_entropy.gif


![교차 엔트로피 애니메이션](./images/anim_02_cross_entropy.gif)

🎬 정답(고양이) 확률을 0.05 → 0.98 로 올렸다. 정답 확률이 **1 에 가까울수록 손실은 0**, **0 에 가까울수록 손실이 급격히 커진다**(확신하고 틀리면 큰 벌점). 오답 클래스의 확률은 $t_k = 0$ 이 곱해져서 손실 계산에 직접 들어가지 않는다.

> **📝 고친 부분** (교재 p.43 · 45 · 메모)
> - 수식의 $\log$ 는 **자연로그**(ln, `np.log`). 교재 예시 $-\log 0.7 \approx 0.357$ 은 자연로그로 맞다(상용로그 `log10` 이면 0.155).
> - 교재 수식 그대로 코딩하면 확률이 0 일 때 $\log 0 = -\infty$ → 실제 코드는 **아주 작은 값(`delta = 1e-7`)을 더해서** 계산한다.
> - 데이터가 여러 개면 합이 아니라 **데이터 개수로 나눈 평균**을 쓴다(sklearn `log_loss` 도 평균).
> - 메모 CEE 1단계의 `t(첫 번째) = 0` 은 앞 문장과 이어지지 않는 줄 → "예: 정답이 [0, 1, 0] 이면 첫 번째 레이블의 t 는 0" 이라는 뜻.

## 4. 이진 분류 — 이진 교차 엔트로피 BCE (교재 p.44)

$$H(y, p) = -\left[\, y\log p + (1-y)\log(1-p) \,\right]$$

- $y$ : 정답 (0 또는 1), $p$ : 시그모이드 출력 = **1 일 확률**
- 정답이 1 이면 $-\log p$, 정답이 0 이면 $-\log(1-p)$ — 둘 중 **하나만** 남는다.

In [11]:
def binary_cross_entropy(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    delta = 1e-7
    return -np.mean(y * np.log(p + delta) + (1 - y) * np.log(1 - p + delta))

# 교재 p.44 예시 : 정답 1, 예측 0.8
print("정답 1, p = 0.8 :", round(binary_cross_entropy(1, 0.8), 4), " = -log(0.8)")
print("정답 0, p = 0.8 :", round(binary_cross_entropy(0, 0.8), 4), " = -log(0.2)  ← 틀린 쪽에 80% → 손실 큼")

# BCE 는 클래스가 [0일 확률, 1일 확률] 2개인 CEE 와 같다
p = 0.8
print("\nCEE 로 계산     :", round(cross_entropy_error([1 - p, p], [0, 1]), 4))

y_true = np.array([1, 0, 1, 1, 0])
p_pred = np.array([0.9, 0.2, 0.6, 0.4, 0.1])
print("데이터 5개 BCE  :", round(binary_cross_entropy(y_true, p_pred), 4),
      "· sklearn", round(log_loss(y_true, p_pred), 4))

정답 1, p = 0.8 : 0.2231  = -log(0.8)
정답 0, p = 0.8 : 1.6094  = -log(0.2)  ← 틀린 쪽에 80% → 손실 큼

CEE 로 계산     : 0.2231
데이터 5개 BCE  : 0.3722 · sklearn 0.3722


> **📝 고친 부분** (교재 p.44)
> - 손실 값 계산 예시가 $= -\log(0.8)$ 에서 끝난다 → 값은 **0.223**. 같은 0.8 이라도 정답이 0 이면 $-\log(0.2) = $ **1.609** 로 7배 크다.

## 5. 손실 함수 실습 — 메모 예시를 신경망 계산으로 이어 보기 (교재 p.49)

메모의 딥러닝 구조 : **입력층 → 은닉층(퍼셉트론 + ReLU) → 출력층(퍼셉트론 + 활성화 함수) → 손실 함수**.
출력층의 입력 신호 총합 $z = w_1x_1 + w_2x_2 + b$ 를 활성화 함수에 넣고, 그 출력과 정답으로 손실을 계산한다.

In [12]:
def softmax(z):
    exp_z = np.exp(z - np.max(z))            # 최댓값을 빼서 오버플로 방지 (결과는 같다)
    return exp_z / np.sum(exp_z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# ① 다중 분류 : 메모의 z = [2.0, 1.0, 0.1] → 소프트맥스 → 교차 엔트로피
z = np.array([2.0, 1.0, 0.1])
y_soft = softmax(z)
print("소프트맥스 출력 :", y_soft.round(2), "· 합", y_soft.sum().round(2))
for k in range(3):
    t_onehot = np.eye(3)[k]
    print(f"  정답이 {k + 1}번째 클래스 {t_onehot.astype(int)} → 손실 {cross_entropy_error(y_soft, t_onehot):.3f}")
print("  → 모델이 가장 높게 본(0.66) 클래스가 정답일 때 손실이 가장 작다")

# ② 이진 분류 : 메모의 p = 0.7 → 클래스 1 일 확률 0.7, 클래스 0 일 확률 0.3
p = 0.7
print(f"\n이진 분류 p = {p} → 정답 1 이면 손실 {binary_cross_entropy(1, p):.3f} · 정답 0 이면 {binary_cross_entropy(0, p):.3f}")
print("  시그모이드 : z = 0.85 →", round(sigmoid(0.85), 3), "(z 가 클수록 1 일 확률이 커진다)")

# ③ 회귀 : 메모 '예측값 = 출력층 퍼셉트론의 출력 (항등 함수라서 WX + b 그대로)'
X = np.array([[1.0, 2.0], [2.0, 1.0], [3.0, 3.0]])   # 데이터 3개, 입력 2개
W = np.array([0.5, 1.0]); b = 0.5
y_reg = X @ W + b                                   # 항등 함수 → 그대로 출력
t_reg = np.array([3.0, 3.0, 4.0])
print("\n회귀 예측 WX + b :", y_reg, "· 정답", t_reg, "→ MSE", round(mean_squared_error(y_reg, t_reg), 4))

소프트맥스 출력 : [0.66 0.24 0.1 ] · 합 1.0
  정답이 1번째 클래스 [1 0 0] → 손실 0.417
  정답이 2번째 클래스 [0 1 0] → 손실 1.417
  정답이 3번째 클래스 [0 0 1] → 손실 2.317
  → 모델이 가장 높게 본(0.66) 클래스가 정답일 때 손실이 가장 작다

이진 분류 p = 0.7 → 정답 1 이면 손실 0.357 · 정답 0 이면 1.204
  시그모이드 : z = 0.85 → 0.701 (z 가 클수록 1 일 확률이 커진다)

회귀 예측 WX + b : [3.  2.5 5. ] · 정답 [3. 3. 4.] → MSE 0.4167


In [13]:
# 메모 : '정답 확률 P3 가 1 에 가까워질수록 -log(P3) 는 0 에 가까워진다 → 0 이 되도록 학습'
t5 = np.array([0, 0, 1, 0, 0])
for p3 in [0.2, 0.5, 0.7, 0.9, 0.99]:
    rest = (1 - p3) / 4
    y5 = np.array([rest, rest, p3, rest, rest])
    print(f"P3 = {p3:<4} → 손실 {cross_entropy_error(y5, t5):.3f}")

P3 = 0.2  → 손실 1.609
P3 = 0.5  → 손실 0.693
P3 = 0.7  → 손실 0.357
P3 = 0.9  → 손실 0.105
P3 = 0.99 → 손실 0.010


- 출력층 활성화 함수와 손실 함수는 **짝**이다 : 항등 함수 ↔ MSE · 시그모이드 ↔ BCE · 소프트맥스 ↔ CEE.
- 손실이 0 이 아닌 이유는 가중치 $W$ · 편향 $b$ 가 아직 데이터에 맞지 않아서 — 다음 노트의 **미분**이 $W$ 를 어느 쪽으로 바꿔야 할지 알려 준다.

## 6. 정리

| | MSE | 교차 엔트로피 (CEE) | 이진 교차 엔트로피 (BCE) |
|---|---|---|---|
| 문제 | 회귀 | 다중 분류 | 이진 분류 |
| 출력층 | 항등 함수 | 소프트맥스 | 시그모이드 |
| 식 | $\frac{1}{n}\sum (y - t)^2$ | $-\sum t_k \log y_k$ | $-[y\log p + (1-y)\log(1-p)]$ |
| 0 이 되는 때 | 예측 = 정답 | 정답 확률 = 1 | 정답 쪽 확률 = 1 |
| 코드 | `mean_squared_error` | `log_loss` | `log_loss` |

- 손실 함수 = 모델이 얼마나 나쁜지를 **숫자 하나**로. 학습 = 이 숫자를 **줄이는 가중치 찾기**.
- 여러 데이터의 손실은 **평균**을 낸다 → 데이터 개수가 달라도 같은 기준으로 비교할 수 있다.
- 다음 노트 : 가중치를 **어느 방향으로** 바꿔야 손실이 줄어드는지 알려 주는 도구 = **미분(기울기)** → [`numerical_differentiation.ipynb`](./numerical_differentiation.ipynb)